# Agentic RAG

## Load Secrets

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## Function (tool) for Reranked Hydrid Search

In [2]:
import os
import cohere
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import FusionQuery, SparseVector, Prefetch

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
)
bm25 = SparseTextEmbedding("Qdrant/bm25")
COLLECTION = "week2_article"


def search_article(query: str) -> dict:
    """Searches the indexed article for any topic.

    Returns up to 5 reranked passages. Call this multiple times with
    different queries when a question spans multiple topics.

    Args:
        query: A natural-language search query.
    """
    dense = co.embed(
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"],
        texts=[query],
    ).embeddings.float[0]

    sv = next(iter(bm25.query_embed([query])))
    sparse = SparseVector(
        indices=sv.indices.tolist(),
        values=sv.values.tolist(),
    )

    cands = qdrant.query_points(
        collection_name=COLLECTION,
        prefetch=[
            Prefetch(query=dense, using="dense", limit=30),
            Prefetch(query=sparse, using="sparse", limit=30),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=30,
    ).points

    rr = co.rerank(
        model="rerank-v3.5",
        query=query,
        documents=[c.payload["text"] for c in cands],
        top_n=10,
    )

    return {
        "results": [
            {"text": cands[r.index].payload["text"], "score": r.relevance_score}
            for r in rr.results
        ]
    }

## Intitialize Agent

In [5]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

agent = LlmAgent(
    name="article_assistant",
    model=LiteLlm("cohere/command-a-03-2025"),
    instruction=(
        "You are a precise reading assistant for an indexed article.\n\n"
        "Use `search_article` to gather facts before answering. Call it multiple "
        "times with different queries for multi-part questions. Quote the article "
        "where helpful. If the article does not cover the answer, say so — "
        "don't invent."
    ),
    tools=[search_article],
)
print("Agent ready.")

Agent ready.


## Run Agent

In [6]:
import asyncio
from google.adk.runners import InMemoryRunner
from google.genai.types import Content, Part

runner = InMemoryRunner(agent, app_name="week2_lab06")


async def ask(question: str, user_id: str = "student") -> None:
    print(f"\n💬 {question}\n")
    session = await runner.session_service.create_session(
        app_name="week2_lab06", user_id=user_id
    )
    async for ev in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(role="user", parts=[Part(text=question)]),
    ):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if getattr(p, "function_call", None):
                    print(f"   🛠  {p.function_call.name}({dict(p.function_call.args)})")
                if getattr(p, "text", None) and ev.is_final_response():
                    print(f"\n🤖 {p.text}")

await ask("Can you tell me the 3 types of agentic memory?")


💬 Can you tell me the 3 types of agentic memory?



c:\Users\Ashutosh Bhadoria\Documents\learn\grokking-agents\.venv\Lib\site-packages\google\adk\tools\function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


   🛠  search_article({'query': '3 types of agentic memory'})

🤖 The article mentions three tiers of agentic memory:
1. **Tier 1 — Short-term / scratchpad memory**: The running context window. The current conversation, the recent tool outputs, the inprogress plan. Cheap, ephemeral, capped by token limits. The engineering problem here is summarization and pruning — keeping the context useful as it grows past your model’s window without losing the threads that matter.
2. **Tier 2 — Session memory**: What the agent remembers within a single user conversation. Usually a rolling summary plus the last N raw turns. The engineering problem is consistency — the summary and the raw history must not contradict each other.
3. **Tier 3 — Long-term memory**: What persists across sessions and users. Vector DBs (semantic), knowledge graphs (relational), episodic stores (event-based). The engineering problem is retrieval — pulling the right memory at the right time without dragging in noise.


## Evaluate

### Single Tool Call

In [7]:
await ask("what are the four flavors of eval?")


💬 what are the four flavors of eval?

   🛠  search_article({'query': 'four flavors of eval'})

🤖 The four flavors of eval are:
1. **Reference-based evals** - Golden inputs with golden outputs. Score with exact match, BLEU/ROUGE for fuzzy text, or LLM-as-judge for nuanced answers. Best for narrow, well-specified tasks.
2. **Trajectory evals** - Score the path, not just the final answer. Did the agent take 3 steps or 30? Did it call the right tools in the right order? Did it loop? This is where most agent-specific eval lives.
3. **End-to-end / online evals** - Run the agent against real user requests in shadow mode and compare outcomes. Slow, expensive, the only thing that catches certain classes of bugs.
4. **Reference-free evals (LLM-as-judge)** - A second LLM scores the agent's output against a rubric. Faster to set up, but you’re trading determinism for coverage. Always validate the judge against human ratings on a sample — judges are biased toward verbose, wellformatted answers and

### Multiple Tool Calls

In [8]:
await ask(
    "Analyse what the pdf says about the model layer and then compare it with the point of view of Agent Harness"
)


💬 Analyse what the pdf says about the model layer and then compare it with the point of view of Agent Harness

   🛠  search_article({'query': 'model layer'})
   🛠  search_article({'query': 'agent harness'})

🤖 The model layer is the small box, and everything else is where the work happens.

The agent harness is the orchestration loop and everything built around the model.

Therefore, the model layer and the agent harness are two different things.


## Recap — naive vs agentic RAG

| | Naive RAG (lab 03) | Agentic RAG (here) |
|---|---|---|
| Who searches? | The pipeline, always once | The LLM, when it decides |
| Multi-hop questions | Single context window | Multiple searches, refined queries |
| Self-correction | None | Yes — re-query on weak hits |
| Latency | Fast, predictable | Slower, variable |
| Cost | One LLM call | N tool turns + final answer |

Agentic RAG isn't always worth it — for simple Q&A, naive RAG is faster and cheaper. **Use agentic when questions are open-ended, multi-part, or when the right query is hard to predict in advance.**